In [ ]:
# ── Mount Google Drive (Colab only) ──
import os

try:
	from google.colab import drive
	from google.colab import auth
	drive.mount('/content/drive')
	auth.authenticate_user()
	IN_COLAB = True
except ModuleNotFoundError:
	IN_COLAB = False
	print("⚠️ Not running in Colab; skipping Drive mount and Colab auth.")
#https://drive.google.com/file/d/1B3ALPLgk4-DHZG7u_P5ppp7zrvQFmVhh/view?usp=drive_link
#https://drive.google.com/file/d/1vHtr4GdB8Gxd83wLIepq9NRhjvtANkoH/view?usp=sharing
FILE_ID = "1vHtr4GdB8Gxd83wLIepq9NRhjvtANkoH"
OUT = "harmonic_1.0.0_data.tar.gz"

downloaded = False

# ── Method 1: gdown (works when not rate-limited) ──
try:
	import gdown
	gdown.download(id=FILE_ID, output=OUT, quiet=False)
	if os.path.exists(OUT) and os.path.getsize(OUT) > 1000:
		downloaded = True
except Exception as e:
	print(f"gdown failed: {e}")

# ── Method 2: Google Drive API (uses Colab auth, bypasses rate limit) ──
if not downloaded and IN_COLAB:
	import time, io
	from googleapiclient.discovery import build
	from googleapiclient.http import MediaIoBaseDownload
	from google.auth import default

	print("⏳ gdown rate-limited — falling back to Google Drive API…\n")

	creds, _ = default()
	service = build('drive', 'v3', credentials=creds)

	# Get file metadata (name + size) for a friendlier progress display
	meta = service.files().get(fileId=FILE_ID, fields='name,size').execute()
	total_bytes = int(meta.get('size', 0))
	file_name = meta.get('name', OUT)

	if total_bytes > 0:
		total_mb = total_bytes / 1e6
		print(f"📦 File: {file_name}  ({total_mb:.1f} MB)\n")
	else:
		print(f"📦 File: {file_name}\n")

	request = service.files().get_media(fileId=FILE_ID)
	start = time.time()

	with open(OUT, 'wb') as f:
		downloader = MediaIoBaseDownload(f, request, chunksize=10*1024*1024)
		done = False
		while not done:
			status, done = downloader.next_chunk()
			if status:
				pct = status.progress() * 100
				dl_bytes = status.resumable_progress
				dl_mb = dl_bytes / 1e6
				elapsed = time.time() - start
				speed = dl_bytes / elapsed if elapsed > 0 else 0
				speed_str = f"{speed/1e6:.1f} MB/s" if speed >= 1e6 else f"{speed/1e3:.0f} KB/s"
				if total_bytes > 0:
					eta = (total_bytes - dl_bytes) / speed if speed > 0 else 0
					eta_str = f"{int(eta//60)}m {int(eta%60)}s" if eta >= 60 else f"{int(eta)}s"
					bar_len = 30
					filled = int(bar_len * pct / 100)
					bar = "█" * filled + "░" * (bar_len - filled)
					print(f"\r  [{bar}] {pct:5.1f}%  {dl_mb:.1f}/{total_mb:.1f} MB  {speed_str}  ETA {eta_str}   ", end="", flush=True)
				else:
					print(f"\r  ↓ {dl_mb:.1f} MB  {speed_str}   ", end="", flush=True)

	elapsed_total = time.time() - start
	print()  # newline after progress bar
	downloaded = True

if downloaded:
	size = os.path.getsize(OUT)
	size_mb = size / 1e6
	print(f"\n✅ Downloaded {OUT} ({size_mb:.1f} MB)")
	if 'elapsed_total' in dir():
		avg_speed = size / elapsed_total if elapsed_total > 0 else 0
		avg_str = f"{avg_speed/1e6:.1f} MB/s" if avg_speed >= 1e6 else f"{avg_speed/1e3:.0f} KB/s"
		print(f"   ⏱ Completed in {elapsed_total:.1f}s (avg {avg_str})")
else:
	print("❌ Download failed. Try again later or download manually from:")
	print(f"   https://drive.google.com/uc?id={FILE_ID}")